In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Building a model
In this section we will look at pratical example of how we can build a regression model to see how whether membrane resistance, spike threshold and capacitance contribute to rheobase. Often we just want to test things whether our measure is different between genotype, treatment, sex, etc. However, we can actually build a more biological plausible model that also gets into some simple causal analysis. This analysis will also cover several techniques such as model comparison, residual analysis and intepretation of regression coefficients.

## Setting the baseline
First we are going to see how well predicted rheobase matches up to the actual rheobase. We can use the simple equation $\Delta V=IR_m$ for the relationship between resistance and rheobase. To get $\Delta V$  we will subtract the resting membrane potential from the spike threshold. To get $I$ we will divide $\Delta V$ by $R_m$. We will regress the predicted rheobase against the actual rebase using a simple linear regression. Then we will analyze the regression output.

### Load and prepare the data
We need to rename our columns so we can use the statsmodels formula API. This means we need to remove spaces and parenthesis for the column names.

In [ ]:
data_path = Path().cwd().parent / "data/stats/msn_data.csv"
df = pd.read_csv(data_path)

df = df.dropna(subset="Vm_B", axis="rows")
df["deltav_mem"] = df["Spike threshold (mV)"] - df["Vm_B"]
df["synth_rheo"] = (df["deltav_mem"] / df["Membrane resistance"]) * 1000
df["mem_res"] = df["Membrane resistance"]
df["rheo"] = df["Rheobase (pA)"]
df["spk_thresh"] = df["Spike threshold (mV)"]

### Run the regression

In [ ]:
model_fit = smf.ols("rheo ~ synth_rheo", data=df).fit()

Plot the resulting fit and the points.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df["synth_rheo"], df["Rheobase (pA)"], ".")
x = np.linspace(df["synth_rheo"].min(), df["synth_rheo"].max(), num=100)
fit = model_fit.params["Intercept"] + model_fit.params["synth_rheo"]*x
ax.plot(x, fit)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

The fit looks goods, but how good is it? Notice how the predicted rheobase seems about 2x larger than the actual rheobase. We can look at the actual model to get more information about this relationship.
### Analyze the regression

In [ ]:
print(model_fit.summary())

Some things to note about the model summary. Our slope coefficent, `synth_rheo` is 0.3509 which means if we compare one cell with a rheobase of say 100 and one with a rheobase of 101 the difference in actual rheobase between the two cells would be 0.3509. This means our input needs to be multiplied by 0.3509 to get to the actual rheobase. This is pretty close to the estimate of the predicted rheobase being 2x larger than the actual rheobase. Why would this be? It probably depends on how we measure membrane resistance. For this data set the cell was held at -70 mV by injecting a slow (DC) current. I regressed the delta v against the current injection amplitude only for acquisition with 150 pA of the 0 pA current injection. We have different sets of ion channels that are active at current injections. *Ih* channels open below -40 mV, Na+ channels start to open with depolarizing current injections, K+ leak channels are probably more important around the resting membrane potential just to name a few. What if we break down the predicted rheobase into its components?

## Begining to breakdown the model
First let's look at how the different components may be related

In [ ]:
columns = [
    "rheo",
    "mem_res",
    "spk_thresh",
    "Vm_B",
    "Cm",
    "Rs_B",
]
fig, ax = plt.subplots(
    nrows=len(columns), ncols=len(columns), layout="constrained", figsize=(15, 15)
)
values = len(columns)
for i in range(values):
    for j in range(values):
        if i != j:
            x = df[columns[i]]
            y = df[columns[j]]
            model_fit = smf.ols(f"{columns[j]} ~ {columns[i]}", data=df).fit()
            x_fit = np.linspace(df[columns[i]].min(), df[columns[i]].max(), num=100)
            y_fit = model_fit.params["Intercept"] + model_fit.params[columns[i]]*x_fit
            ax[i][j].plot(x, y, ".")
            ax[i][j].set_xlabel(columns[i])
            ax[i][j].set_ylabel(columns[j])
            ax[i][j].plot(x_fit,y_fit)
        else:
            ax[i][j].hist(df[columns[i]])
            ax[i][j].set_xlabel(columns[j])

Probably the two biggest things that stand out are that membrane resistance and spike threshold seem to have some relationship with rheobase. The relationship between rheobase and membrane resistance is nonlinear. This makes sense since rheobase and membrane resistance are inversely proportional. Spike threshold is also fairly linearly correlated with spike threshold. While there seem to be some linear relations between other variables the variance around the simple regression line is quite a lot. When we build a regression model with more predictors we want to avoid collinear features since this will impair the regression fit. Also if we look at the histogram distributions we can see that most variables are left skewed (tail to the right).

In [ ]:
plt.plot(1/df["Membrane resistance"], df["Rheobase (pA)"], ".")